# Offline outcome table: review every context level

Builds the (PR × level) outcome table described in roadmap Section 8.5. The frozen reviewer reviews each PR's context levels, AACR-Bench's judge scores each review against the reference comments, and the results go to `data/aacr/outcomes/<run_name>/`.

- **Reviewer prompt:** OpenAI Codex's built-in `/review` rubric, verbatim, as the system message (`configs/review_prompts/codex_review_rubric.md`, from openai/codex at commit `4e4767f`, Apache-2.0). The user message is a one-sentence request followed by the level's `context.md`. Codex's own request tells the reviewer to run `git diff`; a one-call reviewer cannot, so only that sentence is adapted.
- **Reviewer model:** `Qwen/Qwen3.8-27B`, greedy decoding, served by vLLM on the cloud GPU (profile `cloud`).
  - A laptop cannot run it.
  - Profile `local-smoke` runs `Qwen/Qwen3.5-0.8B` in-process on two short levels, only to test the pipeline. Its outputs are **not results**.
  - Profile `sample-local` runs the same small model as reviewer **and** judge on the committed 2-PR sample (`data/aacr_sample/context_levels.json`), on every level up to 100,000 input tokens (what a 39 GB laptop fits), and writes `data/aacr_sample/outcomes/`. It shows the table's real layout; its scores are **not results**.
- **Judge:** AACR-Bench's file-level semantic judge, with its prompt copied verbatim. **The judge model is still an open decision** (roadmap Section 8.5.3). `keyword-test` is a stand-in for pipeline tests only.
- **Inputs:** the one-file context-level dataset built by `scripts/dataset/build_dataset.py data/aacr` (`data/aacr/context_levels.jsonl`, one line per PR with every level's text). Reference comments are read only by the judging stage and never reach the reviewer.
- **Resuming:** re-running reuses every cell whose input, prompt, model and settings are unchanged. Failed calls are retried. Cells skipped as too long are re-checked against the current limit.

**Choose the profile** with the `OUTCOME_PROFILE` environment variable (`cloud` by default), or edit `PROFILE` below.

## Before running the `cloud` profile

Running it uses the cloud GPU, so get the go-ahead first.

1. **Download the reviewer model** on the GPU machine: `.venv/bin/python scripts/setup/download_reviewer_model.py`. It reads the model and revision from `PROFILES["cloud"]` below and downloads that snapshot (about 55.6 GB) through hf-mirror.com into the Hugging Face cache, `/root/autodl-tmp/huggingface` on AutoDL. It never contacts huggingface.co, which is unreachable from mainland China: a request that would go there stops the run with an error, and the log lists every host contacted. The mirror redirects the large weight files to Hugging Face's file CDN (hosts under hf.co), which is reachable there. It then checks every file's hash and that the config and tokenizer load offline, and prints the command for step 2. Re-run it to resume; `--dry-run` shows the plan and the disk check without downloading.
2. **Start the reviewer server** on the GPU machine. Install vLLM 0.17.0 or later in its own environment (`python -m venv /root/vllm-env && /root/vllm-env/bin/pip install vllm`) and activate it (`source /root/vllm-env/bin/activate`); calling its `vllm` by full path fails on the first start, because a build tool it needs (`ninja`) is then not on `PATH`. Then run the command the script prints, for example: `HF_HOME=/root/autodl-tmp/huggingface HF_HUB_OFFLINE=1 HF_ENDPOINT=https://hf-mirror.com vllm serve Qwen/Qwen3.8-27B --revision 37561816e66b5b9153b1b64c10968c18c1e6f4a0 --tokenizer-revision 37561816e66b5b9153b1b64c10968c18c1e6f4a0 --max-model-len 262144 --generation-config vllm --port 8000 --max-num-seqs 8`. `HF_HUB_OFFLINE=1` loads only the downloaded files, and `HF_ENDPOINT` sends anything that still goes online to the mirror. `--generation-config vllm` makes the request's settings the only sampling settings; by default vLLM also applies the model's own generation config. `--max-num-seqs 8` is this notebook's `concurrency`; with vLLM's default of 1,024 the server does not start, because each parallel sequence needs a slot of the model's state cache and about 600 fit on a 96 GB GPU. Levels are capped at 256,000 input tokens, leaving 1,710 for the prompt, 4,096 for the review and some slack. Check the flags against your vLLM version's documentation.
   - **Served, no reviews run yet.** On 2026-11-14 vLLM 0.33.0 served the 27B with this command on one RTX PRO 6000 Blackwell (96 GB): 51 GiB of weights and a cache for 543,646 tokens, which is 2.07 full-length requests. A synthetic prompt of 257,074 tokens was answered in 81 s. Before that, the small `Qwen/Qwen3.5-0.8B` (the same architecture) took 32 real cells through this notebook's code, including one of 255,999 input tokens, and vLLM counted exactly 1,710 prompt tokens more than the dataset's `input_tokens` on every call (the 27B's own chat template measures 1,709), within the 1,710 the level cap leaves.
   - **Speed measured there.** One request alone generates about 26 tokens/s; 8 in parallel generate about 160 tokens/s in total. Prompts are read at about 5,000 tokens/s at 32,000 tokens, 4,300 at 128,000 and 3,200 at 257,000. The full table is 1,569 calls and about 62 million prompt tokens, so time a few PRs first.
   - **Memory.** The cache costs about 64 KB per token (full-attention layers × 2 × key-value heads × head size × 2 bytes = 16 × 2 × 4 × 256 × 2), so about 16 GiB for one full-length request. At startup vLLM logs `GPU KV cache size` and `Maximum concurrency for 262,144 tokens per request`; on a smaller GPU, `--language-model-only` (text only) frees the vision encoder.
   - **Repeatability.** Greedy replies are not guaranteed to repeat: they can change with what else the server is processing and with its cache state. On the 27B, with 8 prompts of about 6,000 tokens, two passes sent one at a time agreed on all 8 once the server had seen the prompts, but the first time the prompts were sent agreed with the second on only 2 of 8. With 8 in parallel, two passes agreed on 7 of 8, and the parallel replies matched the one-at-a-time replies on 2 of 8. Saved cells are reused, so one run is consistent with itself, but a re-run may produce different reviews.
3. **Make the inputs available on that machine:** copy `data/aacr/` (it holds `context_levels.jsonl` and AACR's `positive_samples.json`). The context file is built on a laptop with `scripts/dataset/build_dataset.py data/aacr`, no LLM needed.
4. **Set `base_url` below** if the server is not at `http://localhost:8000/v1`.
5. **Choose and serve the judge model**, then set the `judge` entry. It is still undecided.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "review_outcomes").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from tqdm.auto import tqdm

from src.review_outcomes import backends, table
from src.review_outcomes.parse import PARSER_VERSION
from src.review_outcomes.prompts import (
    JUDGE_PROMPT_VERSION, REVIEW_PROMPT_VERSION, REVIEW_REQUEST, RUBRIC_SHA256, load_rubric, prompt_sha256,
)
from src.tracer_context.aacr import file_sha256

print(PROJECT_ROOT)

## Configuration

In [ ]:
PROFILE = os.environ.get("OUTCOME_PROFILE", "cloud")  # "cloud", "local-smoke" or "sample-local"

DATA_ROOT = PROJECT_ROOT / "data" / "aacr"  # the root given to scripts/dataset/build_dataset.py
DATASET = DATA_ROOT / "positive_samples.json"  # AACR-Bench's file, with the reference comments

PROFILES = {
    "cloud": {
        "run_name": "qwen3.8-27b_codex-rubric_v1",
        "contexts": "data/aacr/context_levels.jsonl", "out_root": "data/aacr/outcomes",
        "entries": None,  # None = every PR built successfully in the contexts file
        "levels": None,   # None = all ten levels
        "reviewer": {"backend": "openai", "base_url": "http://localhost:8000/v1", "model": "Qwen/Qwen3.8-27B",
                     "model_revision": "37561816e66b5b9153b1b64c10968c18c1e6f4a0", "max_tokens": 4096,
                     "enable_thinking": False, "concurrency": 8, "max_input_tokens": 256_000,
                     "api_key_env": "REVIEWER_API_KEY"},
        "judge": {"backend": "openai", "base_url": "http://localhost:8001/v1", "model": "JUDGE_MODEL_TO_BE_DECIDED",
                  "max_tokens": 3000, "enable_thinking": False, "api_key_env": "JUDGE_API_KEY"},
    },
    "local-smoke": {
        "run_name": "smoke_qwen3.5-0.8b",
        "contexts": "data/aacr/context_levels.jsonl", "out_root": "data/aacr/outcomes",
        "entries": ["cline__cline__5177"],  # a small PR; build it first: scripts/dataset/build_dataset.py data/aacr --entries cline__cline__5177
        "levels": ["L01_min", "L02_function"],
        "reviewer": {"backend": "transformers", "model": "Qwen/Qwen3.5-0.8B", "max_tokens": 1024,
                     "enable_thinking": False, "concurrency": 1, "max_input_tokens": 12_000},
        "judge": {"backend": "keyword-test"},
    },
    "sample-local": {  # the committed 2-PR sample; a pipeline demonstration, its scores are not results
        "run_name": "sample_qwen3.5-0.8b",
        "contexts": "data/aacr_sample/context_levels.json", "out_root": "data/aacr_sample/outcomes",
        "entries": None, "levels": None,
        "reviewer": {"backend": "transformers", "model": "Qwen/Qwen3.5-0.8B", "max_tokens": 2048,
                     "enable_thinking": False, "concurrency": 1,
                     "max_input_tokens": 100_000},  # longer levels need more memory than a 39 GB laptop has
        "judge": {"backend": "transformers", "model": "Qwen/Qwen3.5-0.8B", "max_tokens": 1024,
                  "enable_thinking": False},
    },
}
CONFIG = PROFILES[PROFILE]
CONTEXTS = PROJECT_ROOT / CONFIG["contexts"]
OUT = PROJECT_ROOT / CONFIG["out_root"] / CONFIG["run_name"]
print(PROFILE, "->", OUT)

In [ ]:
def make_chat(spec):
    if spec["backend"] == "openai":
        return backends.OpenAIChat(spec["base_url"], spec["model"], api_key=os.environ.get(spec.get("api_key_env", "")),
                                   max_tokens=spec["max_tokens"], enable_thinking=spec.get("enable_thinking"),
                                   model_revision=spec.get("model_revision"))
    if spec["backend"] == "transformers":
        return backends.TransformersChat(spec["model"], max_new_tokens=spec["max_tokens"],
                                         enable_thinking=spec.get("enable_thinking"))
    raise ValueError(f"unknown backend {spec['backend']!r}")


rubric = load_rubric()  # refuses to run if the pinned rubric file was edited
reviewer = make_chat(CONFIG["reviewer"])
print("reviewer:", reviewer.identity)
print("prompt sha256:", prompt_sha256(rubric))

## 1. Find the cells

One cell per (entry, level) in the context-level dataset file. Cells above `max_input_tokens` are recorded as `too_long` instead of being sent.

In [ ]:
cells = table.discover_cells(CONTEXTS, CONFIG["entries"], CONFIG["levels"])
cells_df = pd.DataFrame([{"entry": c.entry, "level": c.level, "input_tokens": c.input_tokens} for c in cells])
limit = CONFIG["reviewer"]["max_input_tokens"]
print(f"{len(cells)} cells from {cells_df.entry.nunique()} entries; "
      f"{int((cells_df.input_tokens > limit).sum())} over {limit} input tokens")
cells_df.groupby("level").input_tokens.agg(["count", "mean", "max"])

## 2. Review

Saved per cell in `reviews/<entry>/<level>.json`: the raw reply, the parsed findings, token counts, the prompt hash and the backend settings. Re-run this cell to resume or to retry errors.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
git_head = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
table.write_json(OUT / "manifest.json", {
    "profile": PROFILE, "config": CONFIG, "written_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "code_commit": git_head, "rubric_sha256": RUBRIC_SHA256, "review_request": REVIEW_REQUEST,
    "prompt_sha256": prompt_sha256(rubric), "review_prompt_version": REVIEW_PROMPT_VERSION,
    "judge_prompt_version": JUDGE_PROMPT_VERSION, "parser_version": PARSER_VERSION,
    "reviewer_identity": reviewer.identity, "dataset": str(DATASET.relative_to(PROJECT_ROOT)),
    "dataset_sha256": file_sha256(DATASET), "contexts": str(CONTEXTS.relative_to(PROJECT_ROOT)),
    "contexts_sha256": file_sha256(CONTEXTS),
})

bar = tqdm(total=len(cells), desc="reviews")
reviews = table.run_reviews(cells, reviewer, rubric, OUT, max_input_tokens=limit,
                            concurrency=CONFIG["reviewer"]["concurrency"], progress=lambda record: bar.update())
bar.close()
pd.Series([r["status"] for r in reviews]).value_counts()

In [ ]:
# Look at one reply and its parsed findings.
sample = next((r for r in reviews if r.get("reply")), None)
if sample:
    reply = sample["reply"]
    print(sample["entry"], sample["level"], sample["status"], f"{reply['completion_tokens']} output tokens,",
          f"{reply['seconds']}s, finish={reply['finish_reason']}")
    print(reply["text"][:3000])
    print(json.dumps(sample["parsed"]["findings"][:3], indent=1))

## 3. Judge

AACR's file-level semantic metric: one judge call per generated comment against that file's reference comments. Malformed replies count as empty reviews. A failed judge call is recorded and retried on the next run.

**The judge model is still open** (roadmap Section 8.5.3). `keyword-test` scores are not meaningful.

In [ ]:
spec = CONFIG["judge"]
if spec["backend"] == "keyword-test":
    judge = backends.KeywordOverlapJudge()
    print("WARNING: keyword-test judge (pipeline test only); scores are not meaningful")
else:
    if "TO_BE_DECIDED" in spec.get("model", ""):
        raise RuntimeError("Choose the judge model first (roadmap Section 8.5.3) and set CONFIG['judge'].")
    judge = backends.judge_caller(make_chat(spec))

references = table.references_by_entry(DATASET)
scorable = sum(r["status"] in ("ok", "malformed", "empty_output") for r in reviews)
bar = tqdm(total=scorable, desc="judging")
judgments = table.run_judgments(reviews, references, judge, OUT, progress=lambda record: bar.update())
bar.close()
print(sum(j["judge_errors"] for j in judgments), "judge errors (re-run this cell to retry them)")

## 4. Outcome table

`table.csv` has one row per (entry, level), and `coverage.csv` one row per reference comment per level. The summary pools counts over PRs, as AACR does, and also shows the per-PR average recall and human-only recall.

In [ ]:
rows, coverage = table.build_table(reviews, judgments, references)
table.write_outputs(OUT, rows, coverage)
print("wrote", OUT / "table.csv", "and", OUT / "coverage.csv")
pd.DataFrame(table.summarize_by_level(rows))

In [ ]:
pd.DataFrame(rows).head(20)